# 05 — Continuous Distribution Calibration

This notebook calibrates the dispersion of the locked probabilistic
model using development out-of-fold predictions only.

Holdout and external-test outcomes are not accessed.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

def locate_repository(start: Path) -> Path:
    current = start.resolve()

    for candidate in (current, *current.parents):
        if (
            candidate
            / "config/"
            "continuous_calibration_spec.yaml"
        ).exists():
            return candidate

    raise FileNotFoundError("Repository root not found.")

ROOT = locate_repository(Path.cwd())

completed = subprocess.run(
    [
        sys.executable,
        str(
            ROOT
            / "tools/"
            "calibrate_selected_oof_distribution.py"
        ),
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)

print(completed.stdout)

if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError(
        "Development-only continuous calibration failed."
    )


 DEVELOPMENT CONTINUOUS CALIBRATION COMPLETE

Selected model: pooled_empirical_residual
Selected family: empirical_residual
Development rows: 152
Development dates: 38

 dispersion_scale  mean_date_crps  standard_error_date_crps  empirical_50_coverage  empirical_80_coverage  empirical_90_coverage  mean_80_width_c  relative_crps_change_vs_identity  within_one_standard_error  strict_crps_winner  selected_scale
             0.50        0.843711                  0.093731               0.190789               0.342105               0.427632         1.161711                         -0.120001                      False               False           False
             0.75        0.788021                  0.085632               0.263158               0.473684               0.598684         1.742566                         -0.046075                      False               False           False
             1.00        0.753313                  0.077587               0.355263               0.63

## Median-preserving dispersion scaling

Let \(Q(	au)\) denote a predictive quantile and let \(m=Q(0.5)\).
For a positive scale \(s\), the calibrated quantile is

\[
\widetilde Q_s(\tau)
=
m+s\{Q(\tau)-m\}.
\]

This transformation changes predictive dispersion without changing
the predictive median. Positive scaling also preserves quantile order.

In [2]:
summary = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "05_calibration_candidate_summary.csv"
)

manifest = json.loads(
    (
        ROOT
        / "data/manifests/"
        "05_continuous_calibration_manifest.json"
    ).read_text(encoding="utf-8")
)

print(
    summary[
        [
            "dispersion_scale",
            "mean_date_crps",
            "standard_error_date_crps",
            "empirical_80_coverage",
            "mean_80_width_c",
            "within_one_standard_error",
            "strict_crps_winner",
            "selected_scale",
        ]
    ].to_string(index=False)
)

 dispersion_scale  mean_date_crps  standard_error_date_crps  empirical_80_coverage  mean_80_width_c  within_one_standard_error  strict_crps_winner  selected_scale
             0.50        0.843711                  0.093731               0.342105         1.161711                      False               False           False
             0.75        0.788021                  0.085632               0.473684         1.742566                      False               False           False
             1.00        0.753313                  0.077587               0.638158         2.323421                      False               False           False
             1.25        0.736409                  0.070476               0.717105         2.904276                       True               False            True
             1.50        0.733092                  0.064641               0.809211         3.485132                       True                True           False
             1.75     

## Selection principle

CRPS remains the sole selection criterion.

Coverage and interval width are reported as diagnostics but do not
determine the selected scale. This avoids replacing a proper scoring
rule with a collection of separate calibration targets.

Among scales within one paired standard error of the strict CRPS
winner, the selected scale is the one closest to one. Thus calibration
is retained only when the development evidence supports changing the
original distribution.

In [3]:
assert manifest["calibration_locked"] is True
assert manifest["holdout_accessed"] is False
assert manifest["external_test_accessed"] is False
assert manifest["market_data_accessed"] is False
assert (
    manifest["maximum_predictive_median_change_c"]
    <= 1.0e-12
)

print(
    "Locked model:",
    manifest["selected_model"],
)

print(
    "Strict CRPS winner scale:",
    manifest["strict_crps_winner_scale"],
)

print(
    "Selected scale:",
    manifest["selected_scale"],
)

print(
    "Calibration applied:",
    manifest["calibration_applied"],
)

Locked model: pooled_empirical_residual
Strict CRPS winner scale: 1.5
Selected scale: 1.25
Calibration applied: True


## Evidential boundary

This stage does not evaluate holdout transfer, June transfer,
market-implied probabilities or trading returns.

Event-probability regularisation is a later and separate operation.